# **Import Utilities**

In [ ]:
import sys
sys.path.append('../..')

from utils.llm_evaluation_utils import *
from utils.prompt_builder import build_rating_prompt, build_critique_prompt
from utils.models_setup import setup_anthropic, query_claude_model
from utils.data_setup import get_dataset_file_path, prepare_project_data

TASK_SUBSET = "1000"           # "all" | "1000" | "50"
PROMPTING_TYPE = "zero_shot"    # "zero_shot" | "few_shot"
STAGE = "ratings"              # or "critiques"
NUM_TRIALS = 1
MAX_TOKENS = 30000
TEMPERATURE = 1

if STAGE == "ratings":
    SYSTEM_MESSAGE = RATING_SYSTEM_MESSAGE
    build_prompt_fn = build_rating_prompt
    update_results_df_fn = update_rating_in_df
    # schedule_file = rating_schedule_file_1000screens
elif STAGE == "critiques":
    SYSTEM_MESSAGE = CRITIQUES_SYSTEM_MESSAGE
    build_prompt_fn = build_critique_prompt
    update_results_df_fn = update_critiques_in_df
    # schedule_file = critiques_schedule_file_1000screens
else:
    raise ValueError(f"Invalid stage: {STAGE}")

# === Model setup ===
client, cfg = setup_anthropic(
    system_message=SYSTEM_MESSAGE,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS
    )

# Load Data

In [ ]:
SELECTED_TASKS = f"{TASK_SUBSET}_screens" if TASK_SUBSET != "all" else "all_tasks"
SELECTED_TASKS += f"-{PROMPTING_TYPE}"

# Batch folders (parameterized by SELECTED_TASKS)
batch_main_dir      = Path(f"./{STAGE}_Batching/Batch_Files-{SELECTED_TASKS}")
prompts_file  = batch_main_dir / f"prompts-{SELECTED_TASKS}.jsonl"

uicrit_file, base64_screens_file, few_shot_samples_file = get_dataset_file_path(STAGE)
uicrit_df = pd.read_parquet(uicrit_file)
base64_screens_df = pd.read_parquet(base64_screens_file)

if few_shot_samples_file:
    few_shot_samples_df = pd.read_parquet(few_shot_samples_file)
else:
    few_shot_samples_df = None

responses_df, results_paths = prepare_project_data(
    model_short=cfg["model_short"],
    num_trials=NUM_TRIALS,
    task_subset=TASK_SUBSET,
    shots=PROMPTING_TYPE,
    stage=STAGE
)
text_responses_jsonl_file = results_paths["results_jsonl"]
model_results_file        = results_paths["results_parquet"]

print("Responses Shape:", responses_df.shape)
responses_df.head(2)

No existing results at '..\..\results\ratings\parquet\ratings-zero_shot-1000_screens-claude4-responses.parquet'. Initializing new DataFrame.
Responses Shape: (1000, 8)


,screen_task_id,screen_id,task,aesthetics_rating,learnability,efficiency,usability_rating,design_quality_rating
0,15_T01,15,Plan and Start Full Body Workouts,None,None,None,None,None
1,28_T01,28,Enter details to Sing In to Scotiabank.,None,None,None,None,None


For testing only

In [ ]:
# few_shot_samples_df = pd.read_parquet(paths["few_shot_samples"])
# # in responses_df use only screen_id from few_shot_samples_df
# responses_df = responses_df[responses_df['screen_id'].isin(few_shot_samples_df['screen_id'])]

# responses_df.head()

# Get Claude API Responses

In [ ]:
query_args = dict(
    client=client,
    model_version=cfg["model_version"],
    max_tokens=MAX_TOKENS,
    temperature=TEMPERATURE,
    system=SYSTEM_MESSAGE,
    image_format="jpeg",
)

run_llm_inference(
    responses_df=responses_df,
    base64_screens_df=base64_screens_df,
    few_shot_samples_df=few_shot_samples_df,
    evaluation_aspects=EVALUATION_MAIN_ASPECTS,
    build_prompt_fn=build_prompt_fn,
    query_fn=query_claude_model,
    query_args=query_args,
    save_jsonl_fn=save_response_text,
    update_results_df_fn=update_rating_in_df,
    output_jsonl=text_responses_jsonl_file,
    guidelines=GUIDELINES,
    prompting_type=PROMPTING_TYPE,
    requests_per_minute=cfg["rpm"],
    stage=STAGE,
)

[360] Processing screen_task_id=26551_T01
[455] Processing screen_task_id=33423_T01
[561] Processing screen_task_id=40882_T01


# Explore Results

In [5]:
responses_df.head()

,screen_task_id,screen_id,task,aesthetics_rating,learnability,efficiency,usability_rating,design_quality_rating
360,26551_T01,26551,Click to play/learn the Guitar chord,6,3,2,4,4
455,33423_T01,33423,Add a Manual Entry for Cycle Racing.,7,4,3,7,7
561,40882_T01,40882,Choose a product to view more information,7,5,5,9,7


In [ ]:
# display_from_index = 0
# index_of_q1 = responses_df.columns.get_loc("Q1")

# responses_df.iloc[display_from_index:, index_of_q1:index_of_q1+15].head()

In [16]:
columns_with_none = (responses_df.isna() | (responses_df == '')).sum()
columns_with_none

screen_task_id           0
screen_id                0
task                     0
aesthetics_rating        0
learnability             0
efficiency               0
usability_rating         0
design_quality_rating    0
dtype: int64

In [17]:
rows_with_none = responses_df[responses_df.isna().any(axis=1)]
rows_with_none

,screen_task_id,screen_id,task,aesthetics_rating,learnability,efficiency,usability_rating,design_quality_rating


# Store Results

In [18]:
responses_df.to_parquet(model_results_file, index=False)